# Retrieval Evaluation — Labels & Regulations, Keyword vs Semantic vs Hybrid

Evaluated **separately** for the two corpora, with the same Hit Rate / MRR
methodology used across the portfolio. This is the first project positioned
to show **hybrid search's real advantage**, because pharmacovigilance queries
are dense with exact identifiers (drug names, MedDRA terms, CFR citations).

## Committed keyword-baseline numbers (memory backend, k=5)

| Corpus | Hit Rate | MRR |
|---|---|---|
| Labels (drug-scoped) | 1.00 | 0.70 |
| Regulations | 0.97 | 0.90 |

Keyword search does well when the query names the drug or citation. **The
gap it can't close is terminology** — see the reaction-only probe below.

## The measured terminology gap (the case for hybrid search)

Reaction-only probes (no drug name), 10 drug/reaction pairs, keyword k=5:

| Query phrasing | Hit rate |
|---|---|
| MedDRA / label term (e.g. "necrotizing fasciitis of the perineum") | **100%** |
| Lay synonym (e.g. "flesh-eating infection of the groin") | **60%** |

That 40-point drop is a *measured* keyword-search failure on paraphrase —
exactly what semantic/hybrid retrieval is expected to recover. Re-run the
semantic and hybrid columns below with `SEARCH_BACKEND=pgvector` (requires
the ingestion pipeline + an OpenAI key) to complete the comparison.

In [1]:
import sys, random
sys.path.append("..")
import pandas as pd
import os
os.environ["LABELS_PATH"] = "../data/labels.csv"
os.environ["REGS_PATH"] = "../data/regulations.csv"
os.environ["SRLC_PATH"] = "../data/srlc_validation.csv"

from pv_assistant.search import MemoryLabelIndex, MemoryRegIndex

## Metrics

In [2]:
def hit_rate(rel): return sum(True in l for l in rel)/len(rel)
def mrr(rel):
    t=0
    for l in rel:
        for r,v in enumerate(l):
            if v: t+=1/(r+1); break
    return t/len(rel)
def evaluate(gt, fn):
    rel=[fn(q["question"], q) for q in gt]
    return {"hit_rate":round(hit_rate(rel),3), "mrr":round(mrr(rel),3)}

## Labels corpus (keyword baseline)

In [3]:
li = MemoryLabelIndex()
gt_labels = pd.read_csv("../data/ground-truth-labels.csv").to_dict("records")
random.seed(0); random.shuffle(gt_labels)

def label_kw(question, q):
    res = li.search(question, num_results=5)
    return [r.get("id")==q["id"] for r in res]

evaluate(gt_labels, label_kw)

{'hit_rate': 1.0, 'mrr': 0.697}

## Regulations corpus (keyword baseline)

In [4]:
ri = MemoryRegIndex()
gt_regs = pd.read_csv("../data/ground-truth-regulations.csv").to_dict("records")
random.seed(0); random.shuffle(gt_regs)

def reg_kw(question, q):
    res = ri.search(question, num_results=5)
    return [r.get("id")==q["id"] for r in res]

evaluate(gt_regs, reg_kw)

{'hit_rate': 0.968, 'mrr': 0.898}

## The terminology gap — reaction-only probes

The honest, isolating test: query by reaction **only** (no drug name), so
retrieval must succeed on terminology alone. MedDRA/label phrasing vs a lay
synonym for the same reaction.

In [5]:
PROBES = [
    ("montelukast",  "suicidal ideation",                     "suicidal thoughts and self-harm"),
    ("ciprofloxacin","aortic aneurysm and dissection",        "a tear or bulge in the main artery"),
    ("canagliflozin","necrotizing fasciitis of the perineum", "flesh-eating infection of the groin"),
    ("febuxostat",   "cardiovascular death",                  "dying from heart problems"),
    ("gabapentin",   "respiratory depression",                "dangerously slowed breathing"),
    ("warfarin",     "hemorrhage",                            "serious internal bleeding"),
    ("atorvastatin", "rhabdomyolysis",                        "severe muscle breakdown"),
    ("sertraline",   "serotonin syndrome",                    "serotonin toxicity"),
    ("lisinopril",   "angioedema",                            "severe swelling of face and throat"),
    ("metformin",    "lactic acidosis",                       "dangerous lactic acid buildup"),
]

def covered(query, drug):
    return any(r.get("drug")==drug for r in li.search(query, num_results=5))

med = sum(covered(m, d) for d,m,_ in PROBES)
lay = sum(covered(l, d) for d,_,l in PROBES)
n = len(PROBES)
print(f"MedDRA/label term phrasing: {med}/{n} = {med/n:.0%}")
print(f"lay/synonym phrasing:       {lay}/{n} = {lay/n:.0%}")

MedDRA/label term phrasing: 10/10 = 100%
lay/synonym phrasing:       6/10 = 60%


**Result: 100% vs 60%.** Keyword search reliably finds the reaction when
the query uses the label's own terminology, and misses 4 of 10 when the
query paraphrases it. This is the evidence-based justification for hybrid
search in this project — not a domain-analogy assumption.

## Semantic + hybrid comparison (run with pgvector)

With `SEARCH_BACKEND=pgvector` and the ingestion pipeline loaded, swap
`MemoryLabelIndex` for `PgHybridIndex("labels")` and call `.search(..., mode=...)`
with `mode` in `{"keyword","semantic","hybrid"}` to fill this table:

| Retrieval | Terminology-gap hit rate (lay phrasing) |
|---|---|
| keyword | 60% (measured above) |
| semantic | _run to fill_ |
| hybrid | _run to fill_ |

The expected story: semantic recovers most of the lay-phrasing misses;
hybrid keeps semantic's paraphrase robustness **and** keyword's exactness on
drug-name / CFR-citation queries. Report the real numbers; don't assume them.

In [7]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [8]:
os.environ.setdefault("SRLC_PATH", "../data/srlc_validation.csv")

from pv_assistant.search import PgHybridIndex
pg_labels = PgHybridIndex("labels")

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
def label_pgvector(question, q, mode):
    res = pg_labels.search(question, num_results=5, mode=mode)
    return [r.get("id") == q["id"] for r in res]

for mode in ["semantic", "hybrid"]:
    result = evaluate(gt_labels, lambda question, q, m=mode: label_pgvector(question, q, m))
    print(f"{mode}: {result}")

semantic: {'hit_rate': 0.988, 'mrr': 0.659}
hybrid: {'hit_rate': 0.988, 'mrr': 0.659}


In [11]:
def covered_pg(query, drug, mode):
    results = pg_labels.search(query, num_results=5, mode=mode)
    return any(r.get("drug") == drug for r in results)

for mode in ["keyword", "semantic", "hybrid"]:
    med = sum(covered_pg(m, d, mode) for d, m, _ in PROBES)
    lay = sum(covered_pg(l, d, mode) for d, _, l in PROBES)
    n = len(PROBES)
    print(f"{mode:10s} MedDRA: {med}/{n} = {med/n:.0%}   lay: {lay}/{n} = {lay/n:.0%}")

keyword    MedDRA: 10/10 = 100%   lay: 2/10 = 20%
semantic   MedDRA: 10/10 = 100%   lay: 9/10 = 90%
hybrid     MedDRA: 10/10 = 100%   lay: 9/10 = 90%
